# VAE Hard Task — Advanced Multi-Modal Clustering

Advanced extensions over Medium Task:
- **6 Models**: MLP-VAE · Beta-VAE · CVAE · Conv1D-VAE · Autoencoder · Multi-Modal VAE
- **2 Baselines**: PCA + KMeans · Spectral (raw features) + KMeans
- **6 Metrics**: Silhouette · Davies-Bouldin · CH · NMI · ARI · Cluster Purity
- **Beta-VAE disentanglement** analysis
- **Reconstruction examples** from VAE latent space
- **Multi-modal fusion**: audio + TF-IDF lyrics + genre one-hot

## 0. Setup

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)

In [ ]:
import warnings
import numpy as np
import torch
from config.config import (
    BANGLA_DIR, BANGLA_QUERIES, BATCH_SIZE, BETA_VAE_B,
    EPOCHS, FMA_METADATA_URL, GENRE_VOCAB, GTZAN_URLS,
    HIDDEN_DIMS, KMEANS_NINIT, LANG_COLORS, LANG_MARKERS,
    LATENT_DIM, LMD_URL, LR, LYRIC_DIM, MODEL_COLORS, N_BANGLA_PER_GENRE,
)
from src.data.bangla import get_bangla
from src.data.fma import download_fma_metadata, load_fma
from src.data.gtzan import download_gtzan_csv, load_gtzan
from src.data.lmd import download_lmd, load_lmd

warnings.filterwarnings("ignore")
np.random.seed(42); torch.manual_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULTS = PROJECT_ROOT / "results" / "hard"
RESULTS.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE)

## 1. Download + Load Datasets

In [ ]:
FMA_DIR   = PROJECT_ROOT / "data" / "fma" / "fma_metadata"
LMD_DIR   = PROJECT_ROOT / "data" / "lmd"
GTZAN_CSV = PROJECT_ROOT / "data" / "gtzan" / "features_30_sec.csv"

download_fma_metadata(FMA_DIR, FMA_METADATA_URL)
download_lmd(LMD_DIR, LMD_URL)
download_gtzan_csv(GTZAN_CSV, GTZAN_URLS)

X_fma, y_fma = load_fma(FMA_DIR)
X_lmd, y_lmd = load_lmd(LMD_DIR)
X_gh,  y_gh  = load_gtzan(GTZAN_CSV)

datasets_raw = []
for X_en, y_en, ds_name in [
    (X_fma, y_fma, "FMA (Free Music Archive)"),
    (X_lmd, y_lmd, "LMD (Lakh MIDI Dataset)"),
    (X_gh,  y_gh,  "GTZAN via GitHub"),
]:
    lang_en = np.array(["English"] * len(X_en))
    bX, by, bl = get_bangla(X_en.shape[1], BANGLA_QUERIES, BANGLA_DIR, N_BANGLA_PER_GENRE)
    if len(bX) > 0:
        X_m = np.vstack([X_en, bX]); y_m = np.concatenate([y_en, by]); l_m = np.concatenate([lang_en, bl])
    else:
        X_m, y_m, l_m = X_en, y_en, lang_en
    datasets_raw.append((X_m, y_m, l_m, ds_name))
    print(f"  {ds_name}: {X_m.shape}")

## 2. Run Full Hard Pipeline

In [ ]:
from scripts.run_hard import full_pipeline

ALL = {}
for X_raw, y_labels, lang_labels, ds_name in datasets_raw:
    ds_key = ds_name.split()[0]
    ALL[ds_key] = full_pipeline(
        X_raw, y_labels, lang_labels, ds_name,
        latent_dim=LATENT_DIM, epochs=EPOCHS,
        beta_vae_b=BETA_VAE_B, device=DEVICE, results_dir=RESULTS,
    )
print("All experiments complete!")

## 3. Metrics Table + Heatmap

In [ ]:
import pandas as pd
from src.visualization.plots import plot_metrics_heatmap

MODEL_KEYS = ["MLP-VAE","Beta-VAE","CVAE","Conv-VAE","AE","Multimodal","PCA","Spectral"]
rows = []
for ds_key, res in ALL.items():
    for mkey in MODEL_KEYS:
        for algo in ["KMeans","Agglomerative","DBSCAN"]:
            r = res["cl"][mkey][algo]
            rows.append({"Dataset": ds_key, "Model": mkey, "Algorithm": algo,
                "Silhouette": r["sil"], "Davies-Bouldin": r["db"],
                "Calinski-H": r["ch"], "NMI": r["nmi"], "ARI": r["ari"], "Purity": r["purity"]})
df_all = pd.DataFrame(rows)
df_all.to_csv(RESULTS / "full_metrics.csv", index=False)
plot_metrics_heatmap(df_all, save_path=RESULTS / "metrics_heatmap.png")
print(df_all[df_all.Algorithm=="KMeans"].to_string(index=False))

## 4. Analysis — Why VAE Outperforms Baselines

In [ ]:
sep = "=" * 70
print(sep)
print("  VAE vs Baselines — KMeans Silhouette")
print(sep)
for ds_key, res in ALL.items():
    pca_sil = res["cl"]["PCA"]["KMeans"]["sil"]
    for mkey in ["MLP-VAE","Beta-VAE","CVAE","Multimodal"]:
        v = res["cl"][mkey]["KMeans"]["sil"]
        delta = v - pca_sil
        verdict = "BETTER" if delta > 0.01 else ("WORSE" if delta < -0.01 else "SIMILAR")
        print(f"  {ds_key:<6} | {mkey:<12} vs PCA: {v:.4f} vs {pca_sil:.4f}  Δ={delta:+.4f}  [{verdict}]")
print()
print("Saved results:", RESULTS)